# Using ProteoBench locally, step by step

ProteoBench is usually used through the web app at https://proteobench.cubimed.rub.de/, but the underlying `proteobench` Python package can also be used directly, without the web app and without any internet connection.

This notebook walks through the core pipeline for one module (`quant_lfq_DDA_ion_QExactive`, DDA precursor-ion quantification on a Q Exactive instrument) using a small example file that already ships with the repository, so it runs offline and in a few seconds:

1. Load a search engine output file
2. Convert it to ProteoBench's standard format
3. Compute per-precursor scores (the "intermediate" table)
4. Summarize those scores into a single datapoint (the metrics ProteoBench reports)
5. Make the same in-depth plots the web app shows

These steps are similar across all modules so far, however they might vary for future modules that are completely different.

**Requirements**: `pip install proteobench` (or an editable install, see the [developer setup guide](https://proteobench.readthedocs.io/en/stable/developer-guide/development-setup.html)). No GitHub token and no network access are needed for anything in this notebook — we never submit or fetch public results here, we only run the scoring pipeline on a local file.

## Step 1: Pick an input file and load it

First step: you give ProteoBench the output file of a search engine / quantification tool, and the name of that tool.

Here we use a small MaxQuant example file bundled with the repository under `test/data/`. If you are running this on your own data, just replace `input_file` and `input_format` (e.g. `"DIA-NN"`, `"FragPipe"`, `"Sage"`, ...).

In [1]:
import os
import warnings

from proteobench.io.parsing.parse_ion import load_input_file

# Path to a small bundled example file (MaxQuant "evidence" output).
# Replace this with the path to your own search engine output file.
input_file = os.path.join(
    "..", "test", "data", "quant", "quant_lfq_ion_DDA_QExactive", "MaxQuant_evidence_sample.txt"
)
input_format = "MaxQuant"

with warnings.catch_warnings():
    # MaxQuant's proforma parsing warns about fixed modifications being applied later; harmless here.
    warnings.simplefilter("ignore")
    input_df = load_input_file(input_file, input_format)

print(f"Loaded {len(input_df)} rows, {input_df.shape[1]} columns")
input_df.head()

Loaded 635 rows, 60 columns


,Sequence,Length,Modifications,Modified sequence,Oxidation (M) Probabilities,Oxidation (M) Score Diffs,Acetyl (Protein N-term),Oxidation (M),Missed cleavages,Proteins,...,Reverse,Potential contaminant,id,Protein group IDs,Peptide ID,Mod. peptide ID,MS/MS IDs,Best MS/MS,AIF MS/MS IDs,Oxidation (M) site IDs
0,ACADAGLLDESFLR,14,Unmodified,_ACADAGLLDESFLR_,NaN,NaN,0,0,0,sp|O95155|UBE4B_HUMAN,...,NaN,NaN,0,563,0,0,0,0.0,NaN,NaN
1,ACADAGLLDESFLR,14,Unmodified,_ACADAGLLDESFLR_,NaN,NaN,0,0,0,sp|O95155|UBE4B_HUMAN,...,NaN,NaN,1,563,0,0,1,1.0,NaN,NaN
2,ACADAGLLDESFLR,14,Unmodified,_ACADAGLLDESFLR_,NaN,NaN,0,0,0,sp|O95155|UBE4B_HUMAN,...,NaN,NaN,2,563,0,0,NaN,NaN,NaN,NaN
3,ACAELHQNVNVK,12,Unmodified,_ACAELHQNVNVK_,NaN,NaN,0,0,0,sp|Q96QK1|VPS35_HUMAN,...,NaN,NaN,3,4796,1,1,2,2.0,NaN,NaN
4,ACAELHQNVNVK,12,Unmodified,_ACAELHQNVNVK_,NaN,NaN,0,0,0,sp|Q96QK1|VPS35_HUMAN,...,NaN,NaN,4,4796,1,1,3,3.0,NaN,NaN


## Step 2: Load the module's parsing settings

Each module ships a set of TOML configuration files (one per supported tool) that describe how to rename columns, map raw files to conditions, flag decoys/contaminants, etc. `ParseSettingsBuilder` reads the right file for the module + tool combination and returns a parser object.

In [2]:
from proteobench.io.parsing.parse_settings import ParseSettingsBuilder
from proteobench.modules.constants import MODULE_SETTINGS_DIRS

module_id = "quant_lfq_DDA_ion_QExactive"

parse_settings_dir = MODULE_SETTINGS_DIRS[module_id]
parser = ParseSettingsBuilder(parse_settings_dir=parse_settings_dir, module_id=module_id).build_parser(input_format)

print("Species in this module:", parser.species_dict())
print("Expected A vs B ratios:", parser.species_expected_ratio())

Species in this module: {'_YEAST': 'YEAST', '_ECOLI': 'ECOLI', '_HUMAN': 'HUMAN'}
Expected A vs B ratios: {'YEAST': {'A_vs_B': 2.0, 'color': '#88ccef'}, 'ECOLI': {'A_vs_B': 0.25, 'color': '#cc6777'}, 'HUMAN': {'A_vs_B': 1.0, 'color': '#ddcc77'}}


## Step 3: Convert to ProteoBench's standard format

`convert_to_standard_format` renames columns, filters decoys, flags contaminants and species, and maps each raw file to its condition ("A" or "B"). It returns the standardized dataframe together with a mapping of condition -> list of raw file names.

In [3]:
prepared_df, replicate_to_raw = parser.convert_to_standard_format(input_df)

print("Raw files per condition:", dict(replicate_to_raw))
prepared_df.head()

Raw files per condition: {'A': ['LFQ_Orbitrap_DDA_Condition_A_Sample_Alpha_01', 'LFQ_Orbitrap_DDA_Condition_A_Sample_Alpha_02', 'LFQ_Orbitrap_DDA_Condition_A_Sample_Alpha_03'], 'B': ['LFQ_Orbitrap_DDA_Condition_B_Sample_Alpha_01', 'LFQ_Orbitrap_DDA_Condition_B_Sample_Alpha_02', 'LFQ_Orbitrap_DDA_Condition_B_Sample_Alpha_03']}


,Sequence,Length,Modifications,Modified sequence,Oxidation (M) Probabilities,Oxidation (M) Score Diffs,Acetyl (Protein N-term),Oxidation (M),Missed cleavages,Proteins,...,MULTI_SPEC,proforma,precursor ion,replicate,LFQ_Orbitrap_DDA_Condition_A_Sample_Alpha_01,LFQ_Orbitrap_DDA_Condition_A_Sample_Alpha_02,LFQ_Orbitrap_DDA_Condition_A_Sample_Alpha_03,LFQ_Orbitrap_DDA_Condition_B_Sample_Alpha_01,LFQ_Orbitrap_DDA_Condition_B_Sample_Alpha_02,LFQ_Orbitrap_DDA_Condition_B_Sample_Alpha_03
0,ACADAGLLDESFLR,14,Unmodified,_ACADAGLLDESFLR_,NaN,NaN,0,0,0,sp|O95155|UBE4B_HUMAN,...,False,ACADAGLLDESFLR,ACADAGLLDESFLR/2,A,True,False,False,False,False,False
1,ACADAGLLDESFLR,14,Unmodified,_ACADAGLLDESFLR_,NaN,NaN,0,0,0,sp|O95155|UBE4B_HUMAN,...,False,ACADAGLLDESFLR,ACADAGLLDESFLR/2,B,False,False,False,False,False,True
2,ACADAGLLDESFLR,14,Unmodified,_ACADAGLLDESFLR_,NaN,NaN,0,0,0,sp|O95155|UBE4B_HUMAN,...,False,ACADAGLLDESFLR,ACADAGLLDESFLR/2,A,False,True,False,False,False,False
3,ACAELHQNVNVK,12,Unmodified,_ACAELHQNVNVK_,NaN,NaN,0,0,0,sp|Q96QK1|VPS35_HUMAN,...,False,ACAELHQNVNVK,ACAELHQNVNVK/3,A,True,False,False,False,False,False
5,ACAELHQNVNVK,12,Unmodified,_ACAELHQNVNVK_,NaN,NaN,0,0,0,sp|Q96QK1|VPS35_HUMAN,...,False,ACAELHQNVNVK,ACAELHQNVNVK/2,A,False,False,True,False,False,False


## Step 4: Compute per-precursor scores (the "intermediate" table)

This is where ProteoBench computes, for every precursor ion, the log2 fold change between condition A and B, and compares it to the expected ratio for its species (`epsilon` = deviation from the expected ratio; smaller is better).

In [4]:
from proteobench.score.quantscoresHYE import QuantScoresHYE

quant_score = QuantScoresHYE("precursor ion", parser.species_expected_ratio(), parser.species_dict())
intermediate = quant_score.generate_intermediate(prepared_df, replicate_to_raw)

print(f"{len(intermediate)} precursor ions scored")
intermediate[["precursor ion", "species", "log2_A_vs_B", "log2_expectedRatio", "epsilon", "nr_observed"]].head()

124 precursor ions scored


,precursor ion,species,log2_A_vs_B,log2_expectedRatio,epsilon,nr_observed
0,ACADAGLLDESFLR/2,HUMAN,-0.812413,0.0,-0.812413,3
1,ACAELHQNVNVK/2,HUMAN,0.272543,0.0,0.272543,3
2,ACAELHQNVNVK/3,HUMAN,NaN,0.0,NaN,3
3,ACALSIEESCRPGDK/3,HUMAN,0.094717,0.0,0.094717,6
4,ACANPAAGSVILLENLR/2,HUMAN,-0.098966,0.0,-0.098966,6


## Step 5: Summarize into a single datapoint

A "datapoint" is the summary that gets shown on the ProteoBench scatter plot: median/mean absolute epsilon, CV quantiles, ROC-AUC, etc., computed at several `min_nr_observed` thresholds (how many raw files a precursor must be seen in). `user_input` holds the metadata you would normally type into the web app's submission form (software name, search parameters, ...).

In [5]:
from proteobench.datapoint.quant_datapoint import QuantDatapointHYE

user_input = {
    "software_name": "MaxQuant",
    "software_version": "1.0",
    "search_engine_version": "1.0",
    "search_engine": "MaxQuant",
    "ident_fdr_psm": 0.01,
    "ident_fdr_peptide": 0.05,
    "ident_fdr_protein": 0.1,
    "enable_match_between_runs": 1,
    "precursor_mass_tolerance": "0.02 Da",
    "fragment_mass_tolerance": "0.02 Da",
    "enzyme": "Trypsin",
    "allowed_miscleavages": 1,
    "min_peptide_length": 6,
    "max_peptide_length": 30,
}

datapoint = QuantDatapointHYE.generate_datapoint(intermediate, input_format, user_input)

# `results` holds the metrics computed at each min_nr_observed threshold (1, 2, 3, ...).
# The web app's slider lets users pick this threshold; the default is 3.
datapoint["results"][3]

{'median_abs_epsilon_global': np.float64(0.2073733169840608),
 'median_abs_epsilon_eq_species': np.float64(0.2264527679809749),
 'median_abs_epsilon_ECOLI': 0.2649746007531384,
 'median_abs_epsilon_HUMAN': 0.18863712189613047,
 'median_abs_epsilon_YEAST': 0.22574658129365588,
 'mean_abs_epsilon_global': np.float64(0.3222922419921658),
 'mean_abs_epsilon_eq_species': np.float64(0.3600250419487489),
 'mean_abs_epsilon_ECOLI': 0.41045098817683545,
 'mean_abs_epsilon_HUMAN': 0.3120354843863225,
 'mean_abs_epsilon_YEAST': 0.3575886532830888,
 'median_log2_empirical_ECOLI': -1.7350253992468616,
 'median_log2_empirical_HUMAN': 0.08680260305747467,
 'median_log2_empirical_YEAST': 1.2073733169840608,
 'median_abs_epsilon_precision_global': np.float64(0.18576821200619875),
 'median_abs_epsilon_precision_eq_species': np.float64(0.1462692115719643),
 'median_abs_epsilon_precision_ECOLI': 0.07927388143825986,
 'median_abs_epsilon_precision_HUMAN': 0.18993044209737242,
 'median_abs_epsilon_precision

## Step 6: Make the same in-depth plots as the web app

The plot generator works directly on the intermediate table from Step 4, so it does not need any data from other workflows or the internet.

In [6]:
from proteobench.plotting.plot_generator_lfq_HYE import LFQHYEPlotGenerator

plot_generator = LFQHYEPlotGenerator()
figures = plot_generator.generate_in_depth_plots(intermediate, parser)

print("Available plots:", list(figures.keys()))
figures["logfc"].show()

Available plots: ['logfc', 'cv', 'ma_plot']


In [7]:
figures["cv"].show()

In [8]:
figures["ma_plot"].show()

## Next steps

- **Using your own data**: replace `input_file`/`input_format` in Step 1 with your own search engine output. Everything downstream stays the same.
- **Using a different module**: change `module_id` in Step 2 to e.g. `"quant_lfq_DIA_ion_Astral"`, and swap `QuantScoresHYE`/`QuantDatapointHYE`/`LFQHYEPlotGenerator` for the classes used by that module (see `proteobench/score/`, `proteobench/datapoint/`, `proteobench/plotting/`, or the corresponding page under `webinterface/pages/`, for what each module uses).
- **Sharing your results with the community**: this notebook only runs the local scoring pipeline. To make your run public and comparable to everyone else's, use the module's page on https://proteobench.cubimed.rub.de/ and its "Submit New Results" tab — see the [Quick Start guide](https://proteobench.readthedocs.io/en/stable/general-information/1-quickstart.html).